# Qwen2.5-7B-Instruct — Product Name Extraction — **Notebook 2: Inference**
### Upload the LoRA adapter from notebook 1, attach it to Qwen2.5-7B-Instruct, run inference

Built for the **NVIDIA RTX PRO 6000 Blackwell (96 GB, sm_120)**.

This is **notebook 2 of 2**. It does **no training**. You upload the `qwen25_7b_lora_adapter.zip` produced by notebook 1, it re-attaches the adapter to a fresh full-fp32 base model, and runs inference on your chunk files one at a time (**BATCH_SIZE = 24**). If Colab disconnects mid-inference, just re-run this notebook, re-upload the adapter, and continue with the remaining chunks — no re-training.

**Exact reproducibility** is preserved: identical pinned stack, identical fp32 + hard-determinism setup, identical base model + tokenizer + adapter, greedy decoding.

**Sections:** 1) install · 2) reproducibility · 3) upload adapter · 4) load base + attach adapter · 5) inference helpers · 6) per-chunk upload → infer → download.

## 1. Install dependencies (exact pinned versions — restarts once)

Installs the **exact** versions requested and then **restarts the session automatically** so the new `torch`/`numpy` binaries are the ones actually imported.

**How to run:** run this cell once → it installs everything and restarts the kernel → when it comes back, **re-run this same cell** (it detects a sentinel file and skips the reinstall) → then run cell **1b** to confirm every version.

Target stack: torch 2.10.0+cu128 / torchvision 0.25.0 / torchaudio 2.10.0, numpy 2.0.2, scipy 1.16.3, scikit-learn 1.6.1, pandas 2.2.2, transformers 4.55.0, trl 0.20.0, peft 0.16.0, accelerate 1.9.0, datasets 3.6.0, openpyxl 3.1.5, sentencepiece 0.2.1.

In [ ]:
# --- Install EXACT pinned versions, then restart the session once ------------
# Target stack (all versions pinned exactly, per request):
#     torch 2.10.0+cu128 / torchvision 0.25.0 / torchaudio 2.10.0  (Blackwell sm_120)
#     numpy 2.0.2 | scipy 1.16.3 | scikit-learn 1.6.1 | pandas 2.2.2
#     transformers 4.55.0 | trl 0.20.0 | peft 0.16.0 | accelerate 1.9.0
#     datasets 3.6.0 | openpyxl 3.1.5 | sentencepiece 0.2.1
#
# Because we change torch and numpy (vs. what Colab preinstalls), the kernel
# MUST restart once so the new binaries are the ones actually imported. This
# cell installs everything, then restarts automatically. After the restart,
# just RE-RUN this cell: it detects the sentinel file and skips reinstalling,
# then falls through to the next cell (1b) for verification.

import os, sys, subprocess

SENTINEL = "/content/.install_done_v3"   # bump this string if you change pins

def sh(cmd):
    print(">>", cmd, flush=True)
    # no -q: full pip output is shown
    subprocess.run(cmd, shell=True, check=False)

if os.path.exists(SENTINEL):
    print("Sentinel found — packages already installed in this session.")
    print("Skipping reinstall. Continue to cell 1b to verify versions.")
else:
    # 1) Torch stack from the cu128 index (Blackwell sm_120 kernels).
    #    torchvision/torchaudio are pinned to the versions that match torch 2.10.0.
    sh(f'{sys.executable} -m pip install '
       f'torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 '
       f'--index-url https://download.pytorch.org/whl/cu128')

    # 2) Scientific stack from PyPI, pinned exactly.
    sh(f'{sys.executable} -m pip install '
       f'"numpy==2.0.2" "scipy==1.16.3" "scikit-learn==1.6.1" "pandas==2.2.2"')

    # 3) Hugging Face training stack + IO libs.
    #    trl 0.20.0 requires transformers>=4.55.0 / accelerate>=1.4.0 / datasets>=3.0.0
    #    -> satisfied by the pins below.
    sh(f'{sys.executable} -m pip install '
       f'"transformers==4.55.0" "trl==0.20.0" "peft==0.16.0" "accelerate==1.9.0" '
       f'"datasets==3.6.0" "openpyxl==3.1.5" "sentencepiece==0.2.1"')

    # Mark done so the post-restart re-run skips the installs above.
    with open(SENTINEL, "w") as f:
        f.write("ok")

    print("\n" + "="*70)
    print("Install complete. RESTARTING the session now so the new torch/numpy")
    print("binaries are the ones imported. After it restarts, RE-RUN THIS CELL")
    print("(it will skip reinstalling), then run cell 1b to verify versions.")
    print("="*70, flush=True)

    # Trigger the restart. (Colab shows 'Your session crashed/restarted' — this
    # is expected and intentional, not an error.)
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)


### 1b. Verify exact versions

Asserts that the pinned packages match exactly (torch 2.10.0+cu128, torchvision 0.25.0, torchaudio 2.10.0, numpy 2.0.2, scipy 1.16.3, scikit-learn 1.6.1, pandas 2.2.2, transformers 4.55.0, trl 0.20.0, peft 0.16.0, accelerate 1.9.0, datasets 3.6.0, openpyxl 3.1.5, sentencepiece 0.2.1), that torch is a `cu128` build with `sm_120` (Blackwell) kernels, and that `sklearn` imports. Run this **after** the install cell has restarted and been re-run.

In [ ]:
# --- Verify the pinned versions are the ones actually loaded ----------------
import platform, importlib

# All packages pinned to exact versions.
EXPECTED_EXACT = {
    "torch"        : "2.10.0",     # checked on leading x.y.z (build tag +cu128 follows)
    "torchvision"  : "0.25.0",
    "torchaudio"   : "2.10.0",
    "numpy"        : "2.0.2",
    "scipy"        : "1.16.3",
    "sklearn"      : "1.6.1",      # scikit-learn imports as "sklearn"
    "pandas"       : "2.2.2",
    "transformers" : "4.55.0",
    "trl"          : "0.20.0",
    "peft"         : "0.16.0",
    "accelerate"   : "1.9.0",
    "datasets"     : "3.6.0",
    "openpyxl"     : "3.1.5",
    "sentencepiece": "0.2.1",
}
EXPECTED_MIN = {}  # (openpyxl / sentencepiece now pinned exactly above)

def _tuple(v):
    parts = v.split("+")[0].split(".")
    out = []
    for p in parts:
        try: out.append(int(p))
        except ValueError: out.append(0)
    return tuple(out)

print(f"Python : {platform.python_version()}  (expected 3.12.13)\n")

problems = []
for mod, want in EXPECTED_EXACT.items():
    try:
        m = importlib.import_module(mod)
        got = getattr(m, "__version__", "?")
        ok = got.split("+")[0] == want
        print(f"{'OK ' if ok else '!! '}{mod:14s} {got:20s} (expected {want})")
        if not ok:
            problems.append(f"{mod}: got {got}, expected {want}")
    except Exception as e:
        print(f"!! {mod:14s} IMPORT FAILED: {e}")
        problems.append(f"{mod}: import failed ({e})")

for mod, want in EXPECTED_MIN.items():
    try:
        m = importlib.import_module(mod)
        got = getattr(m, "__version__", "?")
        ok = _tuple(got) >= want
        print(f"{'OK ' if ok else '!! '}{mod:14s} {got:20s} (expected >= {'.'.join(map(str,want))})")
        if not ok:
            problems.append(f"{mod}: got {got}, expected >= {'.'.join(map(str,want))}")
    except Exception as e:
        print(f"!! {mod:14s} IMPORT FAILED: {e}")
        problems.append(f"{mod}: import failed ({e})")

# CUDA build tag + Blackwell kernels
import torch
print(f"\ntorch CUDA build: {torch.version.cuda}  (expected 12.8)")
if "cu128" not in (torch.__version__ or ""):
    problems.append(f"torch is not a +cu128 build: {torch.__version__}")

# sklearn smoke test (the exact import Section 5 needs)
from sklearn.model_selection import train_test_split
print("sklearn import OK — train_test_split available.")

if torch.cuda.is_available():
    print("\nGPU :", torch.cuda.get_device_name(0),
          "| cc", torch.cuda.get_device_capability(0))
    archs = torch.cuda.get_arch_list()
    print("arch list:", archs)
    if not any("sm_120" in a or "sm_100" in a for a in archs):
        problems.append("torch has NO Blackwell (sm_120) kernels")
    else:
        print("OK: Blackwell (sm_120) kernels present.")
else:
    problems.append("no CUDA GPU detected")

print("\n" + "="*60)
if problems:
    print("VERSION/ENV PROBLEMS DETECTED:")
    for p in problems:
        print("  -", p)
    raise SystemExit(
        "Environment does not match the pinned target. If you just ran the "
        "install cell, make sure you RE-RAN it after the restart. Otherwise "
        "re-run Section 1, let it restart, then re-run it once more."
    )
else:
    print("All versions match the pinned target. Continue to the data cells.")


## 2. Global reproducibility setup (full fp32, hard determinism)

In [ ]:
import os, random, numpy as np, torch

SEED = 42

# --- Deterministic env flags: set BEFORE any CUDA kernels initialize ---------
os.environ["PYTHONHASHSEED"]          = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # required for deterministic cuBLAS matmul

def set_all_seeds(seed: int = SEED, hard: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    # TF32 rounds fp32 matmuls to ~10 bits nondeterministically -> keep OFF for exact fp32.
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32       = False
    # hard=True: raise if ANY op lacks a deterministic implementation (true guarantee).
    # We train in fp32 with stock transformers ops, which all HAVE deterministic kernels,
    # so hard mode should NOT crash here (unlike with Unsloth's fused kernels).
    torch.use_deterministic_algorithms(hard, warn_only=not hard)

set_all_seeds(SEED, hard=True)
print("Seeds + HARD determinism set. CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("\nEXACT-REPRODUCIBILITY MODE: full fp32 compute (no quantization, no 16-bit),")
print("deterministic cuBLAS/cuDNN, TF32 off, fixed seeds. Same GPU -> identical losses.")


## 3. Upload the LoRA adapter from notebook 1

Upload the **`qwen25_7b_lora_adapter.zip`** you downloaded at the end of notebook 1 (~160 MB). This cell unzips it to `/content/qwen25_7b_lora_adapter` and reads its metadata.

In [ ]:
# --- Upload + unzip the adapter produced by notebook 1 -----------------------
import os, zipfile, json, glob
from google.colab import files

WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)

ADAPTER_DIR = "/content/qwen25_7b_lora_adapter"

print("Upload the adapter zip from notebook 1: 'qwen25_7b_lora_adapter.zip'")
_up = files.upload()

_zips = [f for f in _up.keys() if f.lower().endswith(".zip")]
assert _zips, "No .zip uploaded. Re-run this cell and pick qwen25_7b_lora_adapter.zip."
zip_name = _zips[0]
zip_path = zip_name if os.path.exists(zip_name) else os.path.join("/content", zip_name)

# Fresh extraction (clear any previous copy so re-runs are clean).
if os.path.isdir(ADAPTER_DIR):
    import shutil; shutil.rmtree(ADAPTER_DIR)
os.makedirs(ADAPTER_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(ADAPTER_DIR)

# Some zips nest the files one folder deep; flatten if needed.
if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    hits = glob.glob(os.path.join(ADAPTER_DIR, "**", "adapter_config.json"), recursive=True)
    assert hits, "adapter_config.json not found in the uploaded zip."
    ADAPTER_DIR = os.path.dirname(hits[0])

print("\nAdapter extracted to:", ADAPTER_DIR)
print("Files:", sorted(os.listdir(ADAPTER_DIR)))

# Read the base-model metadata notebook 1 wrote (fallback to defaults if absent).
meta_path = os.path.join(ADAPTER_DIR, "finetune_meta.json")
if os.path.exists(meta_path):
    META = json.load(open(meta_path))
else:
    META = {"base_model": "Qwen/Qwen2.5-7B-Instruct", "max_seq_len": 1536, "dtype": "float32"}
print("Adapter meta:", META)


## 4. Load Qwen2.5-7B-Instruct in full fp32 and attach the adapter

Loads the **base** model in full fp32 with eager attention (identical to notebook 1's load), then attaches the uploaded LoRA adapter with `PeftModel.from_pretrained`. No training state, no optimizer, no gradients — so VRAM stays near the model's weight footprint (~30 GB fp32), far below what you saw at the end of training.

In [ ]:
# --- Load base model (full fp32) + attach the trained LoRA adapter -----------
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

MODEL_NAME  = META.get("base_model", "Qwen/Qwen2.5-7B-Instruct")
MAX_SEQ_LEN = int(META.get("max_seq_len", 1536))

# Tokenizer: prefer the one saved with the adapter (guarantees identical vocab/special
# tokens/chat template to notebook 1); fall back to the hub tokenizer if absent.
try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
    print("Loaded tokenizer from the uploaded adapter folder.")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("Loaded tokenizer from the hub.")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base model in full fp32 + eager attention — EXACTLY matching notebook 1's load,
# so attaching the adapter reproduces the fine-tuned model bit-for-bit.
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype        = torch.float32,   # full fp32 (matches training precision)
    attn_implementation = "eager",        # deterministic attention (not SDPA/flash)
    device_map         = {"": 0},
)

# Attach the LoRA adapter. This reproduces the fine-tuned model deterministically.
model = PeftModel.from_pretrained(base, ADAPTER_DIR, torch_dtype=torch.float32)
model.eval()
print("\nAdapter attached. Model dtype:", next(model.parameters()).dtype)
print("Base:", MODEL_NAME, "| adapter:", ADAPTER_DIR)


## 5. Prompt definitions (identical to notebook 1)

Defines `SYSTEM_PROMPT` and `make_user_prompt` — these must be **byte-identical** to notebook 1 so the model sees exactly the same inputs. (Same source cell, copied here.)

In [ ]:
import json

SYSTEM_PROMPT = (
    "You are an expert at extracting product names from announcements about product "
    "launches, introductions, upgrades, enhancements, regulatory approvals.\n"
    "Extract specific product names of the products being launched, introduced, "
    "upgraded, enhanced, or approved. If multiple products in the announcement meet "
    "the criteria, extract these multiple products. Include the exact model or version "
    "of the products if it's given. The product names extracted should have proper "
    "nouns, not just common nouns.\n"
    "Do not extract products that are not being launched, introduced, upgraded, "
    "enhanced, or approved.\n"
    "For each extracted product, include the product's corresponding company "
    "name (e.g. \"Microsoft Windows XP\" instead of just \"Windows XP\"; or "
    "\"Apple iPhone 16\" instead of just \"iPhone 16\"). If the product belongs to "
    "multiple companies, add the respective company names (separated by a slash) to the "
    "product name (e.g. \"Apple / Mercedes-Benz iPod(R) Integration Kit\").\n"
    "If the announcement contains company names but no specific product names, return "
    "an empty array [].\n"
    "Return ONLY a JSON array of strings. If no relevant products are mentioned, "
    "return an empty array []."
)

def make_user_prompt(announcement: str) -> str:
    return (
        "Extract the product names from the following announcement.\n\n"
        f"Announcement:\n{announcement}\n\n"
        "Return only a JSON array of product name strings."
    )

# We use TRL's *prompt-completion* format. With completion_only_loss=True (the
# default for prompt-completion datasets), loss is computed ONLY on the
# completion (the assistant's JSON answer) and the prompt is masked to -100 --
# exactly what Unsloth's train_on_responses_only did, but without needing the
# {% generation %} template markers that Qwen2.5 does not ship.
def to_prompt_completion(announcement: str, products_list):
    target = json.dumps(products_list, ensure_ascii=False)
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": make_user_prompt(announcement)},
    ]
    completion = [
        {"role": "assistant", "content": target},
    ]
    return {"prompt": prompt, "completion": completion}

train_records = [to_prompt_completion(r.announcement, r.products_list) for r in train_df.itertuples()]
eval_records  = [to_prompt_completion(r.announcement, r.products_list) for r in eval_df.itertuples()]

print("Example completion target:", train_records[0]["completion"][0]["content"][:200])


## 6. Inference helpers (run once)

Greedy, deterministic batched generation with **BATCH_SIZE = 24**. Run this once per session; if Colab disconnects, re-run notebook 2 up to here, then continue uploading the remaining chunks.

In [ ]:
# ---- 11a. Inference helpers (run ONCE per session) --------------------------
import time, torch, json, os

import ast, re
import pandas as pd

def parse_products_out(text: str):
    "Parse a model generation into a list[str]; returns [] on failure."
    if not text:
        return []
    s = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    for parser in (json.loads, ast.literal_eval):
        try:
            v = parser(s)
            if isinstance(v, list):
                return [str(p).strip() for p in v if str(p).strip()]
        except Exception:
            pass
    m = re.search(r"\[.*?\]", s, flags=re.DOTALL)
    if m:
        try:
            v = json.loads(m.group(0))
            if isinstance(v, list):
                return [str(p).strip() for p in v if str(p).strip()]
        except Exception:
            pass
    return []



# Deterministic greedy eval mode with KV cache + left padding for batching.
model.eval()
model.config.use_cache = True
tokenizer.padding_side = "left"
gc = model.generation_config
gc.do_sample = False
gc.temperature = None
gc.top_p = None
gc.top_k = None

BATCH_SIZE     = 24     # per request; 7B fp32 on 96 GB handles this comfortably
MAX_NEW_TOKENS = 256

@torch.no_grad()
def generate_batch(announcements):
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user",   "content": make_user_prompt(a)}],
            tokenize=False, add_generation_prompt=True,
        )
        for a in announcements
    ]
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(model.device)
    out = model.generate(
        **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]

def run_inference_on_file(xlsx_path):
    "Read one inference chunk, run greedy batched generation, return the result DataFrame."
    df_in = pd.read_excel(xlsx_path)
    df_in.columns = [c.strip().lower() for c in df_in.columns]
    assert "announcement" in df_in.columns, f"Expected 'announcement' column; got {list(df_in.columns)}"
    df_in["announcement"] = df_in["announcement"].fillna("").astype(str)
    anns = df_in["announcement"].tolist()
    print(f"Rows to process: {len(anns)}")

    raw_outputs = []
    t0 = time.time()
    for start in range(0, len(anns), BATCH_SIZE):
        raw_outputs.extend(generate_batch(anns[start:start + BATCH_SIZE]))
        done = start + min(BATCH_SIZE, len(anns) - start)
        if done % (BATCH_SIZE * 10) == 0 or done == len(anns):
            rate = done / (time.time() - t0)
            eta  = (len(anns) - done) / rate if rate else 0
            print(f"  {done}/{len(anns)}  ({rate:.1f}/s, ETA {eta/60:.1f} min)")
    print(f"Generated {len(raw_outputs)} outputs in {(time.time()-t0)/60:.1f} min.")

    # parse_products_out() was defined in the out-of-sample test section (10).
    parsed = [parse_products_out(o) for o in raw_outputs]
    df_out = df_in.copy()
    df_out["products"]      = [json.dumps(p, ensure_ascii=False) for p in parsed]
    df_out["n_products"]    = [len(p) for p in parsed]
    df_out["raw_model_out"] = raw_outputs
    return df_out

print("Inference helpers ready. BATCH_SIZE =", BATCH_SIZE)
print("Now run the next cell for each of your chunk files (one upload per run).")


### Per-chunk: upload → infer → download (re-run for each file)

Upload one chunk, get `'<name> RESULTS.xlsx'` back, then re-run for the next chunk. Because each announcement is processed independently under greedy decoding, the results are identical regardless of the order you run the chunks (as long as chunk contents and BATCH_SIZE stay fixed).

In [ ]:
# ---- 11b. Upload ONE chunk -> infer -> download. RE-RUN for each file. ------
from google.colab import files as _f

print("Upload ONE inference chunk file, e.g.:")
print("  'Announcements inference sample 1-10,000.xlsx'")
print("  (next run: '...10,001-20,000.xlsx', and so on)")
_up = _f.upload()

_c = [f for f in _up.keys() if f.lower().endswith((".xlsx", ".xls"))]
assert _c, "No .xlsx uploaded. Re-run this cell and pick one chunk file."
chunk_name = _c[0]
chunk_path = chunk_name if os.path.exists(chunk_name) else os.path.join("/content", chunk_name)
print("\nProcessing:", chunk_name)

result_df = run_inference_on_file(chunk_path)

# Output filename mirrors the input so each chunk's result is clearly labeled:
#   'Announcements inference sample 1-10,000.xlsx'
#   -> 'Announcements inference sample 1-10,000 RESULTS.xlsx'
base = os.path.splitext(chunk_name)[0]
out_path = os.path.join(WORK_DIR, f"{base} RESULTS.xlsx")
result_df.to_excel(out_path, index=False)
print("\nSaved:", out_path, "| rows:", len(result_df))
print(result_df["n_products"].describe())

_f.download(out_path)
print(f"\nDownload started: '{base} RESULTS.xlsx'")
print("When done, RE-RUN this cell and upload the NEXT chunk file.")
